<a href="https://colab.research.google.com/github/dvarelaj/nlp-miniproyecto-icesi/blob/main/Sesion4/Sesion4_Mini_Proyecto_4_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini-Proyecto 4: Clasificación de Texto con BERT
**Equipo:** Farid Sandoval, Diana Varela y Daniel García.

## 1. Introducción y Contexto
En este proyecto abordaremos la clasificación de texto utilizando **Transformers**, específicamente la arquitectura **BERT** (Bidirectional Encoder Representations from Transformers). A diferencia de los modelos secuenciales (como LSTM), BERT lee la secuencia completa de palabras a la vez, usando el mecanismo de atención para entender el contexto bidireccional de cada token.

**El Modelo:** Utilizaremos `distilbert-base-uncased`, una versión destilada de BERT que retiene el 97% de sus capacidades de comprensión del lenguaje pero es un 60% más rápido y ligero, ideal para nuestro entorno computacional.

**El Caso de Uso:** En este proyecto predeciremos la **intención del cliente** (Topic Classification) utilizando el dataset `banking77`. Para viabilizar el entrenamiento rápido, filtraremos el dataset a las **10 intenciones bancarias más comunes**.

Para cumplir con los objetivos y comparar enfoques, implementaremos 3 técnicas:
1. **Baseline Clásico:** TF-IDF + Regresión Logística.
2. **Feature Extraction:** DistilBERT congelado (usando los embeddings del token `[CLS]`) + Clasificador Lineal.
3. **Fine-Tuning:** DistilBERT entrenado de extremo a extremo.

<a href="https://colab.research.google.com/github/dvarelaj/nlp-miniproyecto-icesi/blob/main/Entrega.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Instalación de librerías necesarias
!pip install -q datasets transformers evaluate accelerate

# Librerías
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, AutoModel,
    TrainingArguments, Trainer
)
import evaluate

warnings.filterwarnings('ignore')

print("¡Librerías importadas correctamente! GPU disponible:", torch.cuda.is_available())

## 2. Carga de Datos y Análisis Exploratorio (EDA)
Hemos seleccionado el dataset `banking77`, el cual contiene consultas reales de clientes clasificadas por su intención. Esto evita el problema de sesgo natural que presentaban los datasets de análisis de sentimiento en quejas.

Para viabilizar el tiempo de ejecución en este proyecto, nos enfocaremos en las **10 clases mayoritarias**.

In [ ]:
# 1. Cargar el dataset (Banking77 - Consultas bancarias)
print("Descargando dataset banking77...")
dataset = load_dataset("banking77")

# Convertir a Pandas para el EDA y filtrado
df_train = dataset['train'].to_pandas()
df_test = dataset['test'].to_pandas()

# Tomaremos solo el TOP 10 de categorías para agilizar el fine-tuning
top_10_labels = df_train['label'].value_counts().nlargest(10).index
df_train = df_train[df_train['label'].isin(top_10_labels)].copy()
df_test = df_test[df_test['label'].isin(top_10_labels)].copy()

# Re-mapear las etiquetas para que vayan secuencialmente de 0 a 9
label_mapping = {old_label: new_label for new_label, old_label in enumerate(top_10_labels)}
df_train['label'] = df_train['label'].map(label_mapping)
df_test['label'] = df_test['label'].map(label_mapping)

# Obtener los nombres reales de las categorías para las gráficas
nombres_originales = dataset['train'].features['label'].names
nombres_top_10 = [nombres_originales[i] for i in top_10_labels]

print(f"Dataset reducido a {len(df_train)} registros de entrenamiento y {len(df_test)} de prueba.")

# 2. Análisis Exploratorio de Datos (EDA)
plt.figure(figsize=(10, 6))
sns.countplot(data=df_train, y='label', order=df_train['label'].value_counts().index, palette='viridis')
plt.yticks(ticks=range(10), labels=[nombres_top_10[i] for i in df_train['label'].value_counts().index])
plt.title('Distribución de Clases (Top 10 Intenciones Bancarias)')
plt.xlabel('Cantidad de Textos')
plt.ylabel('Intención del Cliente')
plt.tight_layout()
plt.show()

# Ver un ejemplo de texto
print("\nEjemplo de texto del dataset:")
print(f"Texto: '{df_train.iloc[0]['text']}'")
print(f"Intención: {nombres_top_10[df_train.iloc[0]['label']]}")

## 3. Técnica 1: Baseline Clásico (TF-IDF + Regresión Logística)
Para cumplir con el requerimiento de implementar al menos tres técnicas y establecer un punto de comparación sólido, comenzamos con un enfoque de Machine Learning tradicional.

Además, aplicamos el balanceo de clases (`class_weight='balanced'`) desde este primer modelo. De esta forma, aislamos el efecto del manejo del desbalance y podemos medir la verdadera contribución arquitectónica de los Transformers más adelante.

In [ ]:
# Técnica 1: TF-IDF + Logistic Regression
print("Vectorizando textos con TF-IDF...")
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(df_train['text'])
X_test_tfidf = tfidf.transform(df_test['text'])

y_train = df_train['label'].values
y_test = df_test['label'].values

print("Entrenando modelo Baseline con class_weight='balanced'...")
# Usamos Regresión Logística que es el estándar de oro para baselines de texto
baseline_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
baseline_model.fit(X_train_tfidf, y_train)

# Predicción y Evaluación
preds_baseline = baseline_model.predict(X_test_tfidf)
acc_baseline = accuracy_score(y_test, preds_baseline)
f1_baseline = f1_score(y_test, preds_baseline, average='weighted')

print("-" * 40)
print("RESULTADOS BASELINE CLÁSICO")
print("-" * 40)
print(f"Accuracy: {acc_baseline:.4f} ({acc_baseline*100:.2f}%)")
print(f"F1-Score: {f1_baseline:.4f} ({f1_baseline*100:.2f}%)")

## 4. Técnica 2: BERT como Extractor de Características (Congelado)
En este segundo enfoque, utilizamos DistilBERT como un extractor de características. Pasamos los textos por el modelo pre-entrenado **sin actualizar sus pesos** (es decir, congelado).

Extraemos el embedding del token especial `[CLS]`, el cual está diseñado para contener un resumen semántico de toda la oración, y entrenamos un clasificador clásico sobre estos vectores densos. Esta técnica es rápida y nos permite ver qué tanto conocimiento previo (del pre-entrenamiento de BERT) sirve para nuestro dominio bancario sin hacer *fine-tuning*.

In [ ]:
from datasets import Dataset # ¡Esta es la línea que faltaba!

# Técnica 2: Feature Extraction con DistilBERT
print("Cargando modelo y tokenizador de DistilBERT...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_ckpt = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model_fe = AutoModel.from_pretrained(model_ckpt).to(device)

def extract_hidden_states(batch):
    # Tokenizar el texto
    inputs = tokenizer(batch["text"], padding=True, truncation=True, return_tensors="pt").to(device)
    # Extraer características sin calcular gradientes (más rápido y ahorra memoria)
    with torch.no_grad():
        outputs = model_fe(**inputs)
    # Retornar el token [CLS] (índice 0)
    return {"hidden_state": outputs.last_hidden_state[:, 0, :].cpu().numpy()}

# Convertimos nuestros DataFrames a formato Dataset de HuggingFace
hf_train = Dataset.from_pandas(df_train[['text', 'label']])
hf_test = Dataset.from_pandas(df_test[['text', 'label']])

print("Extrayendo embeddings de BERT (esto tomará unos segunditos)...")
hf_train_fe = hf_train.map(extract_hidden_states, batched=True, batch_size=64)
hf_test_fe = hf_test.map(extract_hidden_states, batched=True, batch_size=64)

# Preparamos las matrices para Scikit-Learn
X_train_fe = np.array(hf_train_fe["hidden_state"])
X_test_fe = np.array(hf_test_fe["hidden_state"])

print("Entrenando clasificador sobre los embeddings de BERT...")
lr_clf = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_clf.fit(X_train_fe, y_train)

# Predicción y Evaluación
preds_fe = lr_clf.predict(X_test_fe)
acc_fe = accuracy_score(y_test, preds_fe)
f1_fe = f1_score(y_test, preds_fe, average='weighted')

print("-" * 40)
print("RESULTADOS FEATURE EXTRACTION (BERT Congelado)")
print("-" * 40)
print(f"Accuracy: {acc_fe:.4f} ({acc_fe*100:.2f}%)")
print(f"F1-Score: {f1_fe:.4f} ({f1_fe*100:.2f}%)")

## 5. Técnica 3: Fine-Tuning de DistilBERT


A diferencia del método anterior, aquí **sí actualizaremos los pesos** de todo el modelo Transformer. Le agregamos una "cabeza" de clasificación (una capa densa al final) y entrenamos el modelo de extremo a extremo con un *learning rate* muy bajo (`2e-5`). De esta manera, adaptamos su profunda comprensión semántica específicamente al dominio de consultas bancarias.

In [ ]:
# Técnica 3: Fine-Tuning de DistilBERT
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate

num_labels = 10
print("Cargando modelo DistilBERT con cabeza de clasificación...")
model_ft = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=num_labels).to(device)

# Función de tokenización para el Trainer
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

print("Tokenizando datasets para el entrenamiento...")
tokenized_train = hf_train.map(tokenize_function, batched=True)
tokenized_test = hf_test.map(tokenize_function, batched=True)

# Cargar métricas
metric_acc = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = metric_acc.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    return {"accuracy": acc, "f1": f1}

# Configuración del entrenamiento
training_args = TrainingArguments(
    output_dir="./resultados_banca",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
    report_to="none"
)

# Inicializar el Trainer
trainer = Trainer(
    model=model_ft,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

print("\n Iniciando Fine-Tuning de BERT (Esto tomará unos 2-3 minutos)...")
trainer.train()

print("\nEvaluando el modelo fine-tuneado sobre el set de prueba...")
eval_results = trainer.evaluate()

print("-" * 40)
print("RESULTADOS FINE-TUNING (BERT Completo)")
print("-" * 40)
print(f"Accuracy: {eval_results['eval_accuracy']:.4f} ({eval_results['eval_accuracy']*100:.2f}%)")
print(f"F1-Score: {eval_results['eval_f1']:.4f} ({eval_results['eval_f1']*100:.2f}%)")

## 6. Comparación Final y Análisis de Errores (Innovación)
A continuación presentamos la tabla comparativa estructurada programáticamente en formato Markdown, seguida de un análisis profundo de los errores del modelo ganador para entender en qué casos específicos falla la atención del Transformer.

In [ ]:
from IPython.display import display, Markdown

# Obtener predicciones del modelo fine-tuneado para el análisis
print("Generando predicciones finales para análisis de errores...")
predictions_output = trainer.predict(tokenized_test)
preds_ft = np.argmax(predictions_output.predictions, axis=1)

# 1. Tabla comparativa programática (Atacando la crítica del NB3)
resultados = pd.DataFrame({
    "Modelo": ["TF-IDF + LogReg (Baseline)", "DistilBERT (Congelado)", "DistilBERT (Fine-Tuning)"],
    "Accuracy": [acc_baseline, acc_fe, eval_results['eval_accuracy']],
    "F1-Score": [f1_baseline, f1_fe, eval_results['eval_f1']]
})

# Construir el string en Markdown
markdown_table = "| Modelo | Accuracy | F1-Score |\n|:---|:---:|:---:|\n"
for index, row in resultados.iterrows():
    markdown_table += f"| {row['Modelo']} | {row['Accuracy']:.4f} | {row['F1-Score']:.4f} |\n"

display(Markdown("### Tabla Comparativa de Modelos"))
display(Markdown(markdown_table))

# 2. Gráfica comparativa
plt.figure(figsize=(10, 5))
sns.barplot(x='F1-Score', y='Modelo', data=resultados, palette='magma')
plt.title('Comparación de F1-Score por Técnica')
plt.xlim(0.85, 1.0) # Ajustamos el eje para notar mejor la diferencia
plt.tight_layout()
plt.show()

# 3. Inspección de Errores
print("\n--- INSPECCIÓN DE ERRORES DEL FINE-TUNING ---")
errores_idx = np.where(preds_ft != y_test)[0]
print(f"Total de errores: {len(errores_idx)} de {len(y_test)} textos evaluados.\n")

print("Ejemplos de clasificaciones incorrectas para entender la confusión semántica:")
for i in errores_idx[:3]: # Mostramos los 3 primeros errores
    texto = df_test.iloc[i]['text']
    real_label = nombres_top_10[y_test[i]]
    pred_label = nombres_top_10[preds_ft[i]]
    print(f"\n Texto: '{texto}'")
    print(f" Predijo: {pred_label}")
    print(f" Etiqueta Real: {real_label}")

### Conclusiones Finales

A partir de la experimentación realizada con las tres técnicas, destacamos lo siguiente:

1. **El poder del Baseline:** El modelo clásico de TF-IDF con Regresión Logística y pesos balanceados logró un sorprendente F1-Score inicial (~**93.5%**). Esto demuestra que para textos cortos con vocabularios muy estructurados, el Machine Learning tradicional sigue siendo altamente competitivo y establece una base difícil de superar.
2. **Limitaciones del Feature Extraction:** Usar DistilBERT congelado (~**90.2%**) rindió por debajo del baseline. Al no actualizar sus pesos, el modelo dependió de su conocimiento de lenguaje general, el cual no estaba optimizado para la jerga específica de la banca.
3. **La superioridad del Fine-Tuning:** Al entrenar la red completa, DistilBERT logró superar al baseline (~**94.5%**). El Transformer adaptó sus representaciones internas (embeddings) para entender las sutilezas de los reclamos y consultas bancarias.
4. **Análisis de Errores:** Al inspeccionar las fallas del Fine-Tuning, notamos que no son errores aleatorios, sino confusiones semánticas profundas causadas por el solapamiento de vocabulario. Por ejemplo, en el texto *"My cash withdrawal was partly declined"*, el modelo predice `declined_cash_withdrawal` (guiado por la palabra clave), pero la intención real es `wrong_amount_of_cash_received`. Esto evidencia que el modelo comprende el contexto bancario, pero sufre con las mismas ambigüedades que confundirían a un agente humano en la vida real.

Este flujo de trabajo reafirma que la arquitectura Transformer brilla en todo su potencial cuando le permitimos adaptar su conocimiento pre-entrenado al dominio específico del problema.